# Загрузка данных

In [1]:
import pandas as pd
import json
import os
import pandas as pd
from collections import Counter
import torch
from torch.utils.data import Dataset, DataLoader
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import json
import random
import os
import heapq

In [2]:
program_summary = pd.read_csv("program_summary.csv")
print(program_summary.info())
program_summary.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2598 entries, 0 to 2597
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   title             2598 non-null   object 
 1   description       2594 non-null   object 
 2   level             2598 non-null   object 
 3   goal              2598 non-null   object 
 4   equipment         2597 non-null   object 
 5   program_length    2597 non-null   float64
 6   time_per_workout  2598 non-null   float64
 7   total_exercises   2598 non-null   int64  
 8   created           2597 non-null   object 
 9   last_edit         2596 non-null   object 
dtypes: float64(2), int64(1), object(7)
memory usage: 203.1+ KB
None


,title,description,level,goal,equipment,program_length,time_per_workout,total_exercises,created,last_edit
0,(MASS MONSTER) High Intensity 4 Day Upper Lowe...,Build tones of muscular with this high intensi...,['Intermediate'],"['Muscle & Sculpting', 'Bodyweight Fitness']",Full Gym,12.0,90.0,384,2024-01-20 10:23:00,2025-06-29 12:39:00
1,(NOT MY PROGRAM)SHJ Jotaro,Build strength and size,"['Advanced', 'Intermediate']",['Bodybuilding'],Full Gym,8.0,60.0,224,2024-07-08 02:28:00,2025-06-18 09:15:00
2,1 PowerLift Per Day Powerbuilding 5 Day Bro Split,Based off of Andy Baker's KCS (Kingwood Streng...,"['Beginner', 'Novice', 'Intermediate']","['Athletics', 'Powerlifting', 'Powerbuilding']",Full Gym,6.0,90.0,237,2025-04-23 09:21:00,2025-06-18 11:55:00
3,10 Week Mass Building Program,This workout is designed to increase your musc...,"['Intermediate', 'Advanced']",['Powerbuilding'],Garage Gym,10.0,70.0,280,2024-09-07 03:44:00,2025-06-18 08:01:00
4,10 week deadlift focus,Increase deadlift,"['Intermediate', 'Advanced']","['Powerbuilding', 'Powerlifting', 'Bodybuildin...",Full Gym,10.0,80.0,356,2024-12-23 03:13:00,2025-06-18 12:19:00


In [3]:
programs_detailed = pd.read_csv("programs_detailed_boostcamp_kaggle.csv")
print(programs_detailed.info())

programs_detailed.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 605033 entries, 0 to 605032
Data columns (total 16 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   title                605033 non-null  object 
 1   description          604214 non-null  object 
 2   level                605033 non-null  object 
 3   goal                 605033 non-null  object 
 4   equipment            605017 non-null  object 
 5   program_length       605017 non-null  float64
 6   time_per_workout     605033 non-null  float64
 7   week                 605033 non-null  float64
 8   day                  605033 non-null  float64
 9   number_of_exercises  605033 non-null  float64
 10  exercise_name        605033 non-null  object 
 11  sets                 605033 non-null  float64
 12  reps                 605033 non-null  float64
 13  intensity            605033 non-null  float64
 14  created              605017 non-null  object 
 15  last_edit        

,title,description,level,goal,equipment,program_length,time_per_workout,week,day,number_of_exercises,exercise_name,sets,reps,intensity,created,last_edit
0,Weightlifting Mobility program,This program is designed to help athletes succ...,"['Beginner', 'Novice', 'Intermediate', 'Advanc...","['Olympic Weightlifting', 'Muscle & Sculpting'...",Garage Gym,7.0,50.0,1.0,1.0,16.0,Knee-to-wall ankle dorsiflexion test,3.0,-180.0,6.0,2025-03-09 04:11:00,2025-06-23 03:31:00
1,Weightlifting Mobility program,This program is designed to help athletes succ...,"['Beginner', 'Novice', 'Intermediate', 'Advanc...","['Olympic Weightlifting', 'Muscle & Sculpting'...",Garage Gym,7.0,50.0,1.0,1.0,16.0,Banded ankle distractions,3.0,-18.0,6.0,2025-03-09 04:11:00,2025-06-23 03:31:00
2,Weightlifting Mobility program,This program is designed to help athletes succ...,"['Beginner', 'Novice', 'Intermediate', 'Advanc...","['Olympic Weightlifting', 'Muscle & Sculpting'...",Garage Gym,7.0,50.0,1.0,1.0,16.0,Slant board calf stretch,3.0,-18.0,6.0,2025-03-09 04:11:00,2025-06-23 03:31:00
3,Weightlifting Mobility program,This program is designed to help athletes succ...,"['Beginner', 'Novice', 'Intermediate', 'Advanc...","['Olympic Weightlifting', 'Muscle & Sculpting'...",Garage Gym,7.0,50.0,1.0,1.0,16.0,Seated Tibialis Raise,3.0,15.0,6.0,2025-03-09 04:11:00,2025-06-23 03:31:00
4,Weightlifting Mobility program,This program is designed to help athletes succ...,"['Beginner', 'Novice', 'Intermediate', 'Advanc...","['Olympic Weightlifting', 'Muscle & Sculpting'...",Garage Gym,7.0,50.0,1.0,1.0,16.0,90/90 Hip Rotations,3.0,8.0,6.0,2025-03-09 04:11:00,2025-06-23 03:31:00


# Соединение и просмотр данных.

In [4]:
program_summary['title'] = program_summary['title'].str.strip()
programs_detailed['title'] = programs_detailed['title'].str.strip()

full_df = pd.merge(
    programs_detailed, 
    program_summary[['title', 'goal', 'level', 'equipment']], 
    on='title', 
    how='left'
)

In [5]:
full_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 605033 entries, 0 to 605032
Data columns (total 19 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   title                605033 non-null  object 
 1   description          604214 non-null  object 
 2   level_x              605033 non-null  object 
 3   goal_x               605033 non-null  object 
 4   equipment_x          605017 non-null  object 
 5   program_length       605017 non-null  float64
 6   time_per_workout     605033 non-null  float64
 7   week                 605033 non-null  float64
 8   day                  605033 non-null  float64
 9   number_of_exercises  605033 non-null  float64
 10  exercise_name        605033 non-null  object 
 11  sets                 605033 non-null  float64
 12  reps                 605033 non-null  float64
 13  intensity            605033 non-null  float64
 14  created              605017 non-null  object 
 15  last_edit        

In [6]:
sequences = full_df.groupby(['title', 'week', 'day'])['exercise_name'].apply(list).tolist()
sequences

[['Bench Press (Paused)',
  'Pull-Up (Weighted)',
  'Incline Bench Press (Dumbbell)',
  'Chest Fly (Machine)',
  'Lateral Raise (Dumbbell)',
  'Hammer Curl',
  'Incline Curl (Dumbbell)'],
 ['Squat (Paused)',
  'Good Morning',
  'Bulgarian Split Squat (Dumbbell)',
  'Standing Calf Raise',
  'Leg Extension',
  'Hanging Leg Raise',
  'Upright Row (Barbell)'],
 ['Walk'],
 ['AD Press',
  'Pendlay Row',
  'Pec Fly (Dumbbell)',
  'Lat Pulldown (Single Arm)',
  'Dip (Weighted)',
  'Lateral Raise (Machine)',
  'Bicep Curl (Barbell)'],
 ['Walk'],
 ['Deadlift (Barbell)',
  'Leg Press (45 Degrees)',
  'Leg Extension',
  'Seated Calf Raise',
  'Hip Adductor (Machine)',
  'Hip Abductor (Machine)',
  'Abs Crunch (Weighted)',
  'Face Pull'],
 ['Walk'],
 ['Bench Press (Paused)',
  'Pull-Up (Weighted)',
  'Incline Bench Press (Dumbbell)',
  'Chest Fly (Machine)',
  'Lateral Raise (Dumbbell)',
  'Hammer Curl',
  'Incline Curl (Dumbbell)'],
 ['Squat (Paused)',
  'Good Morning',
  'Bulgarian Split Squat (D

In [7]:
long_sessions = [s for s in sequences if len(s) > 25]
print(f"Количество аномально длинных тренировок (>25): {len(long_sessions)}")
if long_sessions:
    print("Пример аномальной тренировки (первые 10 упражнений):", long_sessions[0][:10])

single_exercises = [s[0] for s in sequences if len(s) == 1]
single_counts = Counter(single_exercises)
print("\nТоп-10 упражнений в тренировках из 1 элемента:")
for ex, count in single_counts.most_common(10):
    print(f"{ex}: {count}")

tuple_seqs = [tuple(s) for s in sequences]
seq_counts = Counter(tuple_seqs)
print(f"\nКоличество уникальных цепочек: {len(seq_counts)}")
print("Топ-5 самых повторяющихся цепочек:")
for seq, count in seq_counts.most_common(5):
    print(f"{count} раз: {seq}")

Количество аномально длинных тренировок (>25): 314
Пример аномальной тренировки (первые 10 упражнений): ['Goblet Squat', 'Glute Bridge (Dumbbell)', 'Dumbbell Row', 'Side Plank', 'Goblet Squat', 'Glute Bridge (Dumbbell)', 'Dumbbell Row', 'Side Plank', 'Goblet Squat', 'Glute Bridge (Dumbbell)']

Топ-10 упражнений в тренировках из 1 элемента:
Run: 492
Walk: 397
Cardio: 268
Deadlift (Barbell): 195
Bench Press (Barbell): 184
Squat (Barbell): 165
Push Up: 137
Cardio (Zone 2): 118
Stretching: 96
Yoga: 86

Количество уникальных цепочек: 13606
Топ-5 самых повторяющихся цепочек:
492 раз: ('Run',)
397 раз: ('Walk',)
268 раз: ('Cardio',)
195 раз: ('Deadlift (Barbell)',)
184 раз: ('Bench Press (Barbell)',)


In [8]:
def clean_my_sequences(raw_sequences):
    processed = []
    
    for seq in raw_sequences:
        collapsed = []
        if len(seq) > 0:
            collapsed.append(seq[0])
            for i in range(1, len(seq)):
                if seq[i] != seq[i-1]:
                    collapsed.append(seq[i])
        
        if 3 <= len(collapsed) <= 20:
            processed.append(collapsed)
            
    return processed

final_sequences = clean_my_sequences(sequences)

print(f"Было цепочек: {len(sequences)}")
print(f"Стало после чистки и схлопывания: {len(final_sequences)}")
print(f"Выкинули мусора: {len(sequences) - len(final_sequences)}")

Было цепочек: 98818
Стало после чистки и схлопывания: 92537
Выкинули мусора: 6281


# Создание словаря

In [9]:
all_exercises = []
for seq in final_sequences:
    all_exercises.extend(seq)

unique_exercises = sorted(list(set(all_exercises)))

vocab = {ex: i + 3 for i, ex in enumerate(unique_exercises)}
vocab['[PAD]'] = 0
vocab['[MASK]'] = 1
vocab['[UNK]'] = 2

tokenized_sequences = [[vocab[ex] for ex in seq] for seq in final_sequences]

class BertTrainDataset(Dataset):
    def __init__(self, sequences, vocab, max_len=20, mask_prob=0.15):
        self.sequences = sequences
        self.vocab = vocab
        self.max_len = max_len
        self.mask_prob = mask_prob

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        tokens = []
        labels = []

        for token in seq:
            prob = random.random()
            if prob < self.mask_prob:
                p = random.random()
                if p < 0.8:
                    tokens.append(self.vocab['[MASK]'])
                elif p < 0.9:
                    tokens.append(random.randint(3, len(self.vocab)-3))
                else:
                    tokens.append(token)
                
                labels.append(token)
            else:
                tokens.append(token)
                labels.append(-100)

        if len(tokens) < self.max_len:
            padding_len = self.max_len - len(tokens)
            tokens = tokens + [self.vocab['[PAD]']] * padding_len
            labels = labels + [-100] * padding_len
        else:
            tokens = tokens[:self.max_len]
            labels = labels[:self.max_len]

        return torch.tensor(tokens), torch.tensor(labels)

train_dataset = BertTrainDataset(tokenized_sequences, vocab)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

print(f"Словарь готов! Уникальных упражнений: {len(unique_exercises)}")
print(f"Данные токенизированы: {len(tokenized_sequences)} тренировок.")

Словарь готов! Уникальных упражнений: 3055
Данные токенизированы: 92537 тренировок.


In [10]:
project_folder = "./gym_bert_data"
if not os.path.exists(project_folder):
    os.makedirs(project_folder)

with open(f'{project_folder}/sequences_final.json', 'w', encoding='utf-8') as f:
    json.dump(final_sequences, f, ensure_ascii=False, indent=4)

# 2. Сохраняем словарь (чтобы ID не "поплыли" при перезапуске)
with open(f'{project_folder}/vocab.json', 'w', encoding='utf-8') as f:
    json.dump(vocab, f, ensure_ascii=False, indent=4)

print(f"Данные записаны в папку '{project_folder}'.")

Данные записаны в папку './gym_bert_data'.


# Обучение.

In [11]:
with open('./gym_bert_data/sequences_final.json', 'r') as f:
    sequences = json.load(f)
with open('./gym_bert_data/vocab.json', 'r') as f:
    vocab = json.load(f)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"🔥 Ракета готова. Двигатель: {device}")

🔥 Ракета готова. Двигатель: mps


In [12]:
class GymDataset(Dataset):
    def __init__(self, sequences, vocab, max_len=20):
        self.sequences = sequences
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        tokens = [self.vocab.get(ex, 2) for ex in self.sequences[idx][:self.max_len]]
        labels = []
        output_tokens = []

        for t in tokens:
            prob = random.random()
            if prob < 0.15:
                output_tokens.append(1)
                labels.append(t)
            else:
                output_tokens.append(t)
                labels.append(-100)
        
        padding_size = self.max_len - len(output_tokens)
        output_tokens += [0] * padding_size
        labels += [-100] * padding_size
        
        return torch.tensor(output_tokens), torch.tensor(labels)

In [13]:
train_size = int(0.9 * len(sequences))
train_seqs = sequences[:train_size]
val_seqs = sequences[train_size:]

train_loader = DataLoader(GymDataset(train_seqs, vocab), batch_size=128, shuffle=True)
val_loader = DataLoader(GymDataset(val_seqs, vocab), batch_size=128)

def get_hit_rate(model, loader, device, k=5):
    model.eval()
    hits = 0
    total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            output = model(x)

            mask = y != -100
            if mask.sum() == 0: continue
            
            predictions = output[mask]
            targets = y[mask]
            
            _, top_k = predictions.topk(k, dim=1)
            hits += (top_k == targets.unsqueeze(1)).any(dim=1).sum().item()
            total += targets.size(0)
            
    return hits / total if total > 0 else 0

In [14]:
class TransformerRec(nn.Module):
    def __init__(self, vocab_size, hidden=256, heads=8, layers=4, max_len=20, dropout=0.2):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, hidden)
        self.pos_emb = nn.Embedding(max_len, hidden)
        self.dropout = nn.Dropout(dropout)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden, 
            nhead=heads, 
            dropout=dropout, 
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=layers)
        
        self.fc = nn.Linear(hidden, vocab_size)

    def forward(self, x):
        positions = torch.arange(x.size(1), device=x.device).unsqueeze(0)
        x = self.dropout(self.token_emb(x) + self.pos_emb(positions))
        x = self.transformer(x)
        return self.fc(x)

In [15]:
model = TransformerRec(len(vocab), dropout=0.2).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)
criterion = nn.CrossEntropyLoss()

num_epochs = 60
best_checkpoints = []
save_dir = './gym_bert_data/checkpoints-2'
os.makedirs(save_dir, exist_ok=True)

for epoch in range(1, num_epochs + 1):
    model.train()
    total_loss = 0
    for i, (batch_x, batch_y) in enumerate(train_loader):
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        output = model(batch_x)
        loss = criterion(output.view(-1, len(vocab)), batch_y.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    val_hit = get_hit_rate(model, val_loader, device, k=5)
    avg_loss = total_loss / len(train_loader)
    
    scheduler.step(val_hit)
    
    print(f"Эпоха {epoch} | Loss: {avg_loss:.4f} | 📊 Hit@5: {val_hit:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")

    checkpoint_path = os.path.join(save_dir, f"gym_bert_v2_ep{epoch}_hit{val_hit:.4f}.pth")
    if len(best_checkpoints) < 5 or val_hit > best_checkpoints[0][0]:
        torch.save(model.state_dict(), checkpoint_path)
        heapq.heappush(best_checkpoints, (val_hit, checkpoint_path))
        if len(best_checkpoints) > 5:
            _, path_to_remove = heapq.heappop(best_checkpoints)
            if os.path.exists(path_to_remove): os.remove(path_to_remove)

print("🏁 Обучение завершено!")

Эпоха 1 | Loss: 5.6429 | 📊 Hit@5: 0.2397 | LR: 0.000100
Эпоха 2 | Loss: 5.0878 | 📊 Hit@5: 0.2720 | LR: 0.000100
Эпоха 3 | Loss: 4.8678 | 📊 Hit@5: 0.2894 | LR: 0.000100
Эпоха 4 | Loss: 4.7352 | 📊 Hit@5: 0.3059 | LR: 0.000100
Эпоха 5 | Loss: 4.6216 | 📊 Hit@5: 0.3104 | LR: 0.000100
Эпоха 6 | Loss: 4.5390 | 📊 Hit@5: 0.3112 | LR: 0.000100
Эпоха 7 | Loss: 4.4562 | 📊 Hit@5: 0.3204 | LR: 0.000100
Эпоха 8 | Loss: 4.3797 | 📊 Hit@5: 0.3299 | LR: 0.000100
Эпоха 9 | Loss: 4.3013 | 📊 Hit@5: 0.3321 | LR: 0.000100
Эпоха 10 | Loss: 4.2370 | 📊 Hit@5: 0.3349 | LR: 0.000100
Эпоха 11 | Loss: 4.1836 | 📊 Hit@5: 0.3414 | LR: 0.000100
Эпоха 12 | Loss: 4.1262 | 📊 Hit@5: 0.3300 | LR: 0.000100
Эпоха 13 | Loss: 4.0667 | 📊 Hit@5: 0.3338 | LR: 0.000100
Эпоха 14 | Loss: 4.0094 | 📊 Hit@5: 0.3340 | LR: 0.000050
Эпоха 15 | Loss: 3.9402 | 📊 Hit@5: 0.3317 | LR: 0.000050
Эпоха 16 | Loss: 3.8905 | 📊 Hit@5: 0.3384 | LR: 0.000050
Эпоха 17 | Loss: 3.8646 | 📊 Hit@5: 0.3357 | LR: 0.000025
Эпоха 18 | Loss: 3.8126 | 📊 Hit@5: 0.336